# Setup NemoClaw and VSS on the Brev instance

This notebook is meant to run **on the Brev instance itself**. It walks through the full Brev-side flow for **NemoClaw / OpenClaw** and the **Video Search and Summarization (VSS)** blueprint: configure or reuse a sandbox, verify the policy and imported skills, open the UI, prepare the host prerequisites needed for local NIM-backed profiles, start the host-side VSS Orchestrator MCP deployment used by the agent, and optionally tear it back down.

**Default launchable:**  
[VSS + NemoClaw launchable](https://brev.nvidia.com/launchable/deploy/now?launchableID=env-3BgcwbtTMrB4IXdnaeDwaq5ULti)

If you are using that launchable, the VM is expected to already have the main prerequisites in place. If not, use the preflight cells below to confirm what is missing.

**What this notebook covers**

1. Set required keys and notebook options.
2. Run preflight checks for the local repo, scripts, policy file, MCP helper, and host prerequisites.
3. Create or reuse the NemoClaw sandbox, configure the NVIDIA Endpoints provider, apply the VSS policy, install VSS skills, and update OpenClaw UI `allowedOrigins`.
4. Verify the live sandbox state, active policy metadata, and workspace-installed skills.
5. Generate an OpenClaw UI link and smoke test chat plus VSS skill visibility.
6. Prepare the host for local NIM-backed VSS profiles by installing/configuring NGC CLI, logging Docker into `nvcr.io`, and preparing the `services/agent/` environment.
7. Start the host-side VSS Orchestrator MCP server, generate compose artifacts for the selected `VSS_PROFILE`, bring the deployment up, and check status.
8. Verify sandbox-to-host reachability through `host.openshell.internal`.
9. Optionally tear the generated VSS deployment down.
10. Optionally uninstall NemoClaw.

**Security:** prefer `NVIDIA_API_KEY` and `NGC_CLI_API_KEY` from environment variables or Brev secrets. Do **not** commit notebook outputs that contain credentials or live access tokens.

## 1. Settings

<span style="color:red"><strong>Important:</strong> set <code>NVIDIA_API_KEY</code> and <code>NGC_CLI_API_KEY</code> here or via the environment.  They are required to run the steps below.</span>

In [ ]:
# ============= Set your NVIDIA API Key here (leave blank to use the existing environment variable) =============
NVIDIA_API_KEY = ""  # nvapi-xxxx
NGC_CLI_API_KEY = ""  # Nvidia Legacy API Key
VSS_PROFILE = "base"  # base, search, alerts, lvs
ALERTS_MODE = "verification"  # verification or real-time; only used when VSS_PROFILE == "alerts"
HARDWARE_PROFILE = "RTXPRO6000BW"
# ===============================================================================================================

In [ ]:
import os
from pathlib import Path

# ================== No need to change these unless you want to customize the scripts ===========================
HOME_DIR = Path.home().resolve()
NVIDIA_API_KEY = NVIDIA_API_KEY or os.environ.get("NVIDIA_API_KEY")
VSS_REPO_DIR = Path(os.environ.get("VSS_REPO_DIR", HOME_DIR / "video-search-and-summarization")).resolve()
DEPLOY_SCRIPTS_DIR = VSS_REPO_DIR / "deploy" / "docker" / "scripts"
NEMOCLAW_INIT_SCRIPT_PATH = DEPLOY_SCRIPTS_DIR / "nemoclaw" / "init_nemoclaw.sh"
NEMOCLAW_REPO_DIR = Path(os.environ.get("NEMOCLAW_REPO_DIR", HOME_DIR / "NemoClaw")).resolve()
NEMOCLAW_INSTALL = NEMOCLAW_REPO_DIR / "install.sh"
POLICY_PATH = VSS_REPO_DIR / "assets" / "vss_nemoclaw_policy.yaml"
OPENCLAW_CONFIG_UPDATE_SCRIPT = DEPLOY_SCRIPTS_DIR / "nemoclaw" / "update_openclaw_config.py"
SKILLS_DIR = VSS_REPO_DIR / "skills"
PLUGIN_DIR = VSS_REPO_DIR / ".openclaw"
AGENT_DIR = VSS_REPO_DIR / "services" / "agent"
MCP_CONFIG_PATH = AGENT_DIR / "src" / "vss_agents" / "orchestrator" / "vss_orchestrator_mcp_config.yml"
ORCHESTRATOR_MCP_HELPER_PATH = DEPLOY_SCRIPTS_DIR / "orchestrator_mcp_helper.py"
ARTIFACT_DIR = VSS_REPO_DIR / ".orchestrator-artifacts"
LOG_PATH = ARTIFACT_DIR / "vss_orchestrator_mcp.log"
UV_BIN_DIR = HOME_DIR / ".local" / "bin" / "uv"
# Set both to the same ID for shared LLM/VLM; set different IDs for dedicated LLM/VLM GPUs.
LOCAL_NIM_LLM_DEVICE_ID = "0"
LOCAL_NIM_VLM_DEVICE_ID = "1"

NGC_CLI_API_KEY = NGC_CLI_API_KEY or os.environ.get("NGC_CLI_API_KEY", "")
HARDWARE_PROFILE = HARDWARE_PROFILE or os.environ.get("HARDWARE_PROFILE", "RTXPRO6000BW")
VSS_PROFILE = VSS_PROFILE or os.environ.get("VSS_PROFILE", "base")
LOCAL_NIM_LLM_DEVICE_ID = str(LOCAL_NIM_LLM_DEVICE_ID or os.environ.get("LOCAL_NIM_LLM_DEVICE_ID", "")).strip()
LOCAL_NIM_VLM_DEVICE_ID = str(LOCAL_NIM_VLM_DEVICE_ID or os.environ.get("LOCAL_NIM_VLM_DEVICE_ID", "")).strip()
MCP_HOST = os.environ.get("VSS_ORCHESTRATOR_MCP_HOST", "0.0.0.0")
MCP_PORT = int(os.environ.get("VSS_ORCHESTRATOR_MCP_PORT", "9902"))
MCP_URL = f"http://{MCP_HOST}:{MCP_PORT}/mcp"
HOST_INTERNAL_ALIAS = "host.openshell.internal"

NEMOCLAW_SANDBOX_NAME = os.environ.get("NEMOCLAW_SANDBOX_NAME", "demo")

print("Sandbox:", NEMOCLAW_SANDBOX_NAME)
print("\nHOME_DIR:", HOME_DIR)
print("VSS_REPO_DIR:", VSS_REPO_DIR)
print("NEMOCLAW_INIT_SCRIPT_PATH:", NEMOCLAW_INIT_SCRIPT_PATH)
print("NEMOCLAW_REPO_DIR:", NEMOCLAW_REPO_DIR)
print("NEMOCLAW_INSTALL:", NEMOCLAW_INSTALL)
print("POLICY_PATH:", POLICY_PATH)
print("OPENCLAW_CONFIG_UPDATE_SCRIPT:", OPENCLAW_CONFIG_UPDATE_SCRIPT)
print("SKILLS_DIR:", SKILLS_DIR)
print("PLUGIN_DIR:", PLUGIN_DIR)
print("AGENT_DIR:", AGENT_DIR)
print("MCP_CONFIG_PATH:", MCP_CONFIG_PATH)
print("ORCHESTRATOR_MCP_HELPER_PATH:", ORCHESTRATOR_MCP_HELPER_PATH)
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("LOG_PATH:", LOG_PATH)
print("VSS_PROFILE:", VSS_PROFILE)
print("HARDWARE_PROFILE:", HARDWARE_PROFILE)
print("LOCAL_NIM_LLM_DEVICE_ID:", LOCAL_NIM_LLM_DEVICE_ID or "(profile default)")
print("LOCAL_NIM_VLM_DEVICE_ID:", LOCAL_NIM_VLM_DEVICE_ID or "(profile default)")
print("HOST_INTERNAL_ALIAS:", HOST_INTERNAL_ALIAS)
print("MCP_URL:", MCP_URL)
print("NGC_CLI_API_KEY set:", bool((NGC_CLI_API_KEY or "").strip()))
print("NVIDIA_API_KEY set:", bool((NVIDIA_API_KEY or "").strip()))


## 2. Preflight

Run the next cell to confirm the expected keys, files, commands are present on the instance.


In [ ]:
import importlib.util
import os
import shutil
from pathlib import Path

RED = "\033[31m"
RESET = "\033[0m"
GREEN = "\033[32m"
YELLOW = "\033[33m"
uv_path = Path(UV_BIN_DIR)

helper_spec = importlib.util.spec_from_file_location("orchestrator_mcp_helper", ORCHESTRATOR_MCP_HELPER_PATH)
if helper_spec is None or helper_spec.loader is None:
    raise ImportError(f"Could not load MCP helper from {ORCHESTRATOR_MCP_HELPER_PATH}")
orchestrator_mcp_helper = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(orchestrator_mcp_helper)

OrchestratorTool = orchestrator_mcp_helper.OrchestratorTool
build_vss_ui_url = orchestrator_mcp_helper.build_vss_ui_url
poll_compose_op = orchestrator_mcp_helper.poll_compose_op
require_success = orchestrator_mcp_helper.require_success
tool_call = orchestrator_mcp_helper.tool_call

if not shutil.which("uv") and uv_path.exists():
    os.environ["PATH"] = f"{uv_path.parent}:{os.environ.get('PATH', '')}"

openshell_cli = shutil.which("openshell") or shutil.which("nemoclaw")
nemoclaw_install_fallback = NEMOCLAW_INSTALL.is_file() and os.access(NEMOCLAW_INSTALL, os.X_OK)

required_checks = {
    "init_nemoclaw.sh": NEMOCLAW_INIT_SCRIPT_PATH.is_file(),
    "OpenShell/NemoClaw CLI or install fallback": openshell_cli is not None or nemoclaw_install_fallback,
    "vss_nemoclaw_policy.yaml": POLICY_PATH.is_file(),
    "update_openclaw_config.py": OPENCLAW_CONFIG_UPDATE_SCRIPT.is_file(),
    "skills/": SKILLS_DIR.is_dir(),
    ".openclaw/package.json": (PLUGIN_DIR / "package.json").is_file(),
    "services/agent/": AGENT_DIR.is_dir(),
    "vss_orchestrator_mcp_config.yml": MCP_CONFIG_PATH.is_file(),
    "orchestrator_mcp_helper.py": ORCHESTRATOR_MCP_HELPER_PATH.is_file(),
    "docker": shutil.which("docker") is not None,
    "openshell": shutil.which("openshell") is not None,
    "npm": shutil.which("npm") is not None,
    "python3": shutil.which("python3") is not None,
    "uv": shutil.which("uv") is not None,
    "NVIDIA_API_KEY set": bool((NVIDIA_API_KEY or "").strip()),
    "NGC_CLI_API_KEY set": bool((NGC_CLI_API_KEY or "").strip()),
}

for label, ok in required_checks.items():
    status = "OK " if ok else "NO "
    color = GREEN if ok else RED
    print(f"{color}{status}{RESET} {label}")


## 3. Install and Configure NemoClaw (OpenClaw) for VSS skills

The next cell runs the script in the foreground so you can watch progress.

The script will:

- create or update the NemoClaw sandbox,
- configure the OpenShell provider for the NVIDIA Endpoints-backed model,
- apply the VSS NemoClaw policy,
- install VSS skills,
- update the OpenClaw UI `allowedOrigins` setting.

In [ ]:
import os
import shutil
import subprocess
import time
from collections import deque
from pathlib import Path

init_path = Path(NEMOCLAW_INIT_SCRIPT_PATH)
if not init_path.is_file():
    raise FileNotFoundError(f"Missing init script: {init_path}")

if not NVIDIA_API_KEY.strip():
    raise RuntimeError(
        "NVIDIA_API_KEY is required. Set it in the settings cell or export it before running this cell."
    )

if not POLICY_PATH.is_file():
    raise FileNotFoundError(f"Missing policy file: {POLICY_PATH}")

if not OPENCLAW_CONFIG_UPDATE_SCRIPT.is_file():
    raise FileNotFoundError(f"Missing OpenClaw config update script: {OPENCLAW_CONFIG_UPDATE_SCRIPT}")

# Ensure Node 22+ / npm are available — init_nemoclaw.sh's plugin install step calls `npm pack`.
if shutil.which("npm") is None:
    print("npm not found; installing nvm + Node 22 ...", flush=True)
    install_node_script = """
set -e
export NVM_DIR="$HOME/.nvm"
if [ ! -s "$NVM_DIR/nvm.sh" ]; then
  curl -fsSL https://raw.githubusercontent.com/nvm-sh/nvm/v0.40.1/install.sh | bash
fi
. "$NVM_DIR/nvm.sh"
nvm install 22
nvm alias default 22
node --version
npm --version
"""
    subprocess.run(["bash", "-lc", install_node_script], check=True)
    # Make the freshly installed node visible to this notebook process.
    nvm_node_root = Path.home() / ".nvm" / "versions" / "node"
    if nvm_node_root.is_dir():
        installed = sorted((p for p in nvm_node_root.iterdir() if p.is_dir()), reverse=True)
        if installed:
            os.environ["PATH"] = f"{installed[0] / 'bin'}:{os.environ.get('PATH', '')}"
    print("npm now at:", shutil.which("npm"), flush=True)
else:
    print("npm already available at:", shutil.which("npm"), flush=True)

env = os.environ.copy()
env["VSS_REPO_DIR"] = str(VSS_REPO_DIR)
env["NEMOCLAW_REPO_DIR"] = str(NEMOCLAW_REPO_DIR)
env["NEMOCLAW_SANDBOX_NAME"] = NEMOCLAW_SANDBOX_NAME
env["NVIDIA_API_KEY"] = NVIDIA_API_KEY.strip()
env["NEMOCLAW_POLICY_FILE"] = str(POLICY_PATH)
env["OPENCLAW_CONFIG_UPDATE_SCRIPT"] = str(OPENCLAW_CONFIG_UPDATE_SCRIPT)

timeout_sec = 3600
last_output_lines = deque(maxlen=40)
start = time.monotonic()

print("Running:", init_path, flush=True)
print("Using NemoClaw repo:", NEMOCLAW_REPO_DIR, flush=True)
print("Applying policy file:", POLICY_PATH, flush=True)
print("Using OpenClaw config update script:", OPENCLAW_CONFIG_UPDATE_SCRIPT, flush=True)
process = subprocess.Popen(
    ["bash", str(init_path)],
    env=env,
    cwd=str(init_path.parent),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    if process.stdout is None:
        raise RuntimeError("Failed to capture init_nemoclaw.sh output")

    for line in process.stdout:
        print(line, end="", flush=True)
        last_output_lines.append(line.rstrip("\n"))
        if time.monotonic() - start > timeout_sec:
            process.kill()
            process.wait()
            raise TimeoutError(f"init_nemoclaw.sh timed out after {timeout_sec} seconds")
finally:
    if process.stdout is not None:
        process.stdout.close()

returncode = process.wait()
if returncode != 0:
    tail = "\n".join(last_output_lines)
    raise RuntimeError(
        f"init_nemoclaw.sh exited with {returncode}\n"
        f"Last output:\n{tail}"
    )
print("Done.", flush=True)


## 4. Verify sandbox, policy, and skills

Rerun the next cell after the install cell. It performs a live verification pass and prints a fresh UTC timestamp so you can tell the output is current.

The next cell checks:

- whether the sandbox exists,
- the current active sandbox policy metadata,
- the expected local policy path and version,
- which user-added (non-bundled) OpenClaw skills are visible,
- the installed OpenClaw plugins (`plugins list` / `plugins doctor`) and the full skills list, to confirm the VSS plugin install succeeded.


In [ ]:
from datetime import datetime, timezone
import json
import re
import shlex
import socket
import subprocess
from pathlib import Path


def run(cmd, check=False, echo=True):
    print("$", shlex.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True, check=check)
    if echo:
        if r.stdout:
            print(r.stdout)
        if r.stderr:
            print(r.stderr)
    return r


print("Verification time (UTC):", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S %Z"))
print("Sandbox:", NEMOCLAW_SANDBOX_NAME)
print("Expected policy file:", POLICY_PATH)

policy_text = Path(POLICY_PATH).read_text()
policy_version_match = re.search(r"^version:\s*(\d+)", policy_text, re.MULTILINE)
print("Expected local policy version:", policy_version_match.group(1) if policy_version_match else "unknown")

sandbox_result = run(["openshell", "sandbox", "get", NEMOCLAW_SANDBOX_NAME], echo=False)
sandbox_summary = "\n".join(part for part in (sandbox_result.stdout, sandbox_result.stderr) if part)
sandbox_summary = re.sub(r"\x1b\[[0-9;]*m", "", sandbox_summary)
phase_match = re.search(r"Phase:\s+(.+)", sandbox_summary)
namespace_match = re.search(r"Namespace:\s+(.+)", sandbox_summary)
sandbox_id_match = re.search(r"Id:\s+(.+)", sandbox_summary)
print("Sandbox namespace:", namespace_match.group(1).strip() if namespace_match else "unknown")
print("Sandbox phase:", phase_match.group(1).strip() if phase_match else "unknown")
print("Sandbox id:", sandbox_id_match.group(1).strip() if sandbox_id_match else "unknown")

policy_result = run(["openshell", "policy", "get", NEMOCLAW_SANDBOX_NAME])

policy_summary = "\n".join(part for part in (policy_result.stdout, policy_result.stderr) if part)
status_match = re.search(r"Status:\s+(.+)", policy_summary)
active_match = re.search(r"Active:\s+(.+)", policy_summary)
hash_match = re.search(r"Hash:\s+([0-9a-f]+)", policy_summary)
print("Active policy status:", status_match.group(1).strip() if status_match else "unknown")
print("Active policy version:", active_match.group(1).strip() if active_match else "unknown")
print("Active policy hash:", hash_match.group(1) if hash_match else "unknown")

gateway_container = subprocess.run(
    ["docker", "ps", "--format", "{{.Names}}"], capture_output=True, text=True, check=True
).stdout.splitlines()
gateway_container = next((name for name in gateway_container if name.startswith("openshell-cluster-")), None)
print("Gateway container:", gateway_container)

if gateway_container:
    print("User-added OpenClaw skills (not bundled):")
    skills_cmd = [
        "sudo",
        "docker",
        "exec",
        gateway_container,
        "kubectl",
        "exec",
        "-n",
        "openshell",
        NEMOCLAW_SANDBOX_NAME,
        "-c",
        "agent",
        "--",
        "sh",
        "-lc",
        'su - sandbox -c "openclaw skills list --json"',
    ]
    print("$", shlex.join(skills_cmd))
    skills_result = subprocess.run(skills_cmd, capture_output=True, text=True, check=True)
    skills_text = skills_result.stdout.strip() or skills_result.stderr.strip()
    skills_payload = json.loads(skills_text)
    skills = skills_payload["skills"] if isinstance(skills_payload, dict) else skills_payload

    user_added_skills = [s for s in skills if not s.get("bundled", False)]
    if user_added_skills:
        for skill in user_added_skills:
            print(f"- {skill['name']}")
    else:
        print("No user-added OpenClaw skills found.")
else:
    print("Could not determine the OpenShell gateway container; user-added skills were not checked.")

# --- Verify the openclaw plugins install path: plugins list / doctor / skills list ---
if gateway_container:
    def _in_sandbox_show(label, sandbox_cmd):
        full = [
            "sudo", "docker", "exec", gateway_container,
            "kubectl", "exec", "-n", "openshell", NEMOCLAW_SANDBOX_NAME, "-c", "agent", "--",
            "sh", "-lc", f"su - sandbox -c {shlex.quote(sandbox_cmd)}",
        ]
        print(f"\n=== {label} ===")
        print("$", shlex.join(full))
        r = subprocess.run(full, capture_output=True, text=True)
        if r.stdout:
            print(r.stdout)
        if r.stderr:
            print(r.stderr)
        return r

    _in_sandbox_show("openclaw plugins list", "openclaw plugins list")
    _in_sandbox_show("openclaw plugins doctor", "openclaw plugins doctor")
    # _in_sandbox_show("openclaw skills list", "openclaw skills list")


## 5. Open the UI and smoke test it

Run the next cell to print a fresh **OpenClaw UI** link for the current Brev environment.

### Test 1. Open the OpenClaw UI

Use the **OpenClaw UI** link printed by the next cell.

Open that link in your browser.


In [ ]:
import os
import re
import subprocess


def read_etc_environment():
    env = {}
    try:
        with open("/etc/environment", encoding="utf-8") as fp:
            for raw in fp:
                line = raw.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                env[key.strip()] = value.strip().strip('"').strip("'")
    except OSError:
        return env
    return env


gateway_container = subprocess.run(
    ["docker", "ps", "--format", "{{.Names}}"], capture_output=True, text=True, check=True
).stdout.splitlines()
gateway_container = next((name for name in gateway_container if name.startswith("openshell-cluster-")), None)
print("Gateway container:", gateway_container)

if not gateway_container:
    raise RuntimeError("Could not determine the OpenShell gateway container; no UI link generated.")

openclaw_ui_url = None
env_id = os.environ.get("BREV_ENV_ID", "").strip() or read_etc_environment().get("BREV_ENV_ID", "").strip()
if env_id:
    origin = f"https://openclaw0-{env_id}.brevlab.com"
    token_result = subprocess.run(
        ["nemoclaw", NEMOCLAW_SANDBOX_NAME, "gateway-token", "--quiet"],
        capture_output=True,
        text=True,
        check=True,
    )
    token_result = token_result.stdout.strip()
    openclaw_ui_url = f"{origin}/#token={token_result}" if token_result else origin
    print("OpenClaw UI:", openclaw_ui_url)
else:
    print(
        "Could not resolve Brev environment ID. Set BREV_ENV_ID in the environment "
        "or add it to /etc/environment."
    )

### Test 2. Verify the LLM works in chat

Start a chat with the agent and send a simple prompt such as:

- `hello`
- `what model are you using?`
- `list your available skills`

![OpenClaw UI link screenshot](./images/OpenClawUIChat.png)

You should get a normal model response back. If the UI opens but chat fails, re-check provider setup and the active policy.

### Test 3. Verify the VSS skills are imported

Open the **Skills** tab in the UI and confirm the **VSS skills** are present.

![OpenClaw UI link screenshot](./images/OpenClawUISkills.png)


## 6. Prepare the host for local NIM-backed VSS profiles

If you plan to deploy local NIM-backed VSS profiles through the orchestrator tools, complete the next three pre-steps on the host first:

1. install and configure the NGC CLI,
2. authenticate Docker to `nvcr.io`, and
3. prepare the `services/agent/` Python environment.

### 6.1 Install and configure NGC CLI

This pre-step checks whether `ngc` is already installed, installs it if needed, and writes the local NGC CLI config using `NGC_CLI_API_KEY`.

In [ ]:
import os
import platform
import shutil
import subprocess


def run(cmd: str) -> str:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{result.stderr}\n{result.stdout}")
    return result.stdout.strip()


ngc_path = shutil.which("ngc")
if ngc_path:
    version = run("ngc --version 2>&1 | head -1")
    print(f"NGC CLI already installed: {version}")
else:
    arch = platform.machine()
    filename = "ngccli_linux_arm64.zip" if arch in ("aarch64", "arm64") else "ngccli_linux.zip"
    ngc_cli_version = "4.13.0"
    url = (
        "https://api.ngc.nvidia.com/v2/resources/nvidia/ngc-apps/ngc_cli/"
        f"versions/{ngc_cli_version}/files/{filename}"
    )

    print(f"Installing NGC CLI {ngc_cli_version}...")
    run(f"cd /tmp && wget -q --content-disposition '{url}' -O ngc_cli.zip")

    size = os.path.getsize("/tmp/ngc_cli.zip")
    if size < 1000:
        raise RuntimeError(
            f"NGC CLI download failed: /tmp/ngc_cli.zip is only {size} bytes."
        )

    run("cd /tmp && unzip -o ngc_cli.zip")
    run("sudo cp -r /tmp/ngc-cli/* /usr/local/bin/")
    run("rm -rf /tmp/ngc_cli.zip /tmp/ngc-cli")

    version = run("ngc --version 2>&1 | head -1")
    print(f"Installed NGC CLI: {version}")


if not NGC_CLI_API_KEY:
    raise RuntimeError(
        "NGC_CLI_API_KEY is not set. Set it in the notebook settings cell or export it "
        "in the environment before running this step."
    )

print("Configuring NGC CLI...")
ngc_dir = os.path.expanduser("~/.ngc")
os.makedirs(ngc_dir, exist_ok=True)

with open(os.path.join(ngc_dir, "config"), "w") as f:
    f.write(f""";WARNING - This is a machine generated file. Do not edit manually.
;WARNING - To update local config settings, see 'ngc config set -h'.

[CURRENT]
apikey = {NGC_CLI_API_KEY}
format_type = ascii
org = nvstaging
""")

print("NGC CLI configured.")
print(run("ngc config current"))

### 6.2 Docker login to `nvcr.io`

If you plan to deploy local NIM-backed VSS profiles, authenticate Docker to the NVIDIA Container Registry before using the orchestrator deployment tools.

This pre-step runs `docker login nvcr.io` with `NGC_CLI_API_KEY`.

In [ ]:
import os
import subprocess

ngc_cli_api_key = os.environ.get("NGC_CLI_API_KEY") or NGC_CLI_API_KEY
if not ngc_cli_api_key:
    raise RuntimeError("NGC_CLI_API_KEY is not set. Export it before running this cell.")

login_result = subprocess.run(
    [
        "docker",
        "login",
        "nvcr.io",
        "--username",
        "$oauthtoken",
        "--password",
        ngc_cli_api_key,
    ],
    capture_output=True,
    text=True,
)
if login_result.returncode != 0:
    raise RuntimeError(f"Docker login to nvcr.io failed\n{login_result.stderr}")

print("Docker login to nvcr.io: OK")

### 6.3 Prepare the `services/agent/` environment

The orchestrator MCP server is part of the VSS agent package, so install the Python environment in `services/agent/` before starting the server.

This runs `uv sync` from the agent directory.

In [ ]:
import os
import shutil
import subprocess

if shutil.which("uv") is None:
    raise RuntimeError("uv is not installed. Install uv first, then re-run this cell.")

uv_env = os.environ.copy()
uv_env.pop("VIRTUAL_ENV", None)
subprocess.run(["uv", "sync"], cwd=str(AGENT_DIR), env=uv_env, check=True)
venv_info = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "-c",
        "import os, sys; print('uv VIRTUAL_ENV:', os.environ.get('VIRTUAL_ENV'))",
    ],
    cwd=str(AGENT_DIR),
    env=uv_env,
    check=True,
    capture_output=True,
    text=True,
)
print(venv_info.stdout.strip())
print("Environment is ready.")

## 7. Start the VSS Orchestrator MCP server and deploy a profile

Run the next cells in order.

This section keeps each action separate:

1. prepare helper functions,
2. start the host-side VSS Orchestrator MCP server,
3. generate compose artifacts for the selected `VSS_PROFILE`,
4. bring the deployment up,
5. check compose status, and
6. optionally tear the deployment down.

For full details, profile-specific overrides, manual MCP usage from OpenClaw, and teardown steps, refer to [`deploy_vss_orchestrator_mcp.ipynb`](./deploy_vss_orchestrator_mcp.ipynb).

### 7.1 Start the VSS Orchestrator MCP server

Run the next cell to stop any previously recorded MCP server from this notebook session and start a fresh one.

In [ ]:
import importlib.util
import shutil
import signal
import subprocess
import sys
import time


helper_spec = importlib.util.spec_from_file_location("orchestrator_mcp_helper", ORCHESTRATOR_MCP_HELPER_PATH)
if helper_spec is None or helper_spec.loader is None:
    raise ImportError(f"Could not load MCP helper from {ORCHESTRATOR_MCP_HELPER_PATH}")
orchestrator_mcp_helper = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(orchestrator_mcp_helper)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

for label, ok in {
    "uv": shutil.which("uv") is not None,
    "docker": shutil.which("docker") is not None,
    "openshell": shutil.which("openshell") is not None,
    "agent dir": AGENT_DIR.is_dir(),
    "MCP config": MCP_CONFIG_PATH.is_file(),
}.items():
    if not ok:
        raise RuntimeError(f"Missing prerequisite: {label}. See deploy_vss_orchestrator_mcp.ipynb for the full setup.")


def _wait_for_mcp_health(process: subprocess.Popen, timeout_s: int = 60, interval_s: int = 3) -> None:
    deadline = time.time() + timeout_s
    last_error = "health check did not run"
    while time.time() < deadline:
        return_code = process.poll()
        if return_code is not None:
            raise RuntimeError(f"MCP server exited before becoming healthy with exit code {return_code}: {last_error}")
        try:
            healthy, message = orchestrator_mcp_helper.check_mcp_health(MCP_URL, AGENT_DIR)
        except subprocess.TimeoutExpired:
            healthy, message = False, "health command timed out"
        last_error = message
        if healthy:
            print(f"MCP health check passed: {message}")
            return
        time.sleep(interval_s)

    raise RuntimeError(f"MCP server did not become healthy within {timeout_s}s: {last_error}")


existing_pid = globals().get("VSS_ORCHESTRATOR_MCP_PID")
if existing_pid:
    try:
        os.kill(existing_pid, signal.SIGTERM)
        print(f"Stopped existing MCP server PID {existing_pid}")
        time.sleep(2)
    except ProcessLookupError:
        print(f"Recorded MCP server PID {existing_pid} is no longer running")

env = os.environ.copy()
env.setdefault("PYTHONUNBUFFERED", "1")
log_handle = LOG_PATH.open("w")
process = subprocess.Popen(
    [
        "uv",
        "run",
        "nat",
        "mcp",
        "serve",
        "--config_file",
        str(MCP_CONFIG_PATH),
        "--host",
        MCP_HOST,
        "--port",
        str(MCP_PORT),
    ],
    cwd=str(AGENT_DIR),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,
)
VSS_ORCHESTRATOR_MCP_PID = process.pid
print(f"Started MCP server with PID {VSS_ORCHESTRATOR_MCP_PID}")
_wait_for_mcp_health(process)
print("MCP log:", LOG_PATH)
print("MCP URL:", MCP_URL)


### 7.2 Check orchestrator profiles and prerequisites

This confirms the MCP server is responding, validates host prerequisites through the orchestrator, and records the profiles supported by the current deployment config.

In [ ]:
profiles_result = require_success(
    tool_call(OrchestratorTool.PROFILES, mcp_url=MCP_URL, agent_dir=AGENT_DIR),
    "profiles",
)
require_success(tool_call(OrchestratorTool.PREREQS, mcp_url=MCP_URL, agent_dir=AGENT_DIR), "prereqs")

supported_profiles: set[str] = set()
for key in ("profiles", "supported_profiles"):
    value = profiles_result.get(key)
    if isinstance(value, list):
        for item in value:
            if isinstance(item, str):
                supported_profiles.add(item)
            elif isinstance(item, dict):
                name = item.get("name") or item.get("profile") or item.get("id")
                if isinstance(name, str) and name:
                    supported_profiles.add(name)

print("Supported profiles:", sorted(supported_profiles))

### 7.3 Generate compose artifacts for the selected `VSS_PROFILE`

This uses the current `VSS_PROFILE`, `HARDWARE_PROFILE`, and any available API keys from the settings cell.

In [ ]:
selected_profile = VSS_PROFILE
if supported_profiles and selected_profile not in supported_profiles:
    raise RuntimeError(
        f"Unsupported VSS_PROFILE '{selected_profile}'. Supported profiles: {sorted(supported_profiles)}"
    )

print("Selected VSS_PROFILE:", selected_profile)
if supported_profiles:
    print("Supported profiles:", sorted(supported_profiles))

env_overrides = [f"HARDWARE_PROFILE={HARDWARE_PROFILE}"]
if LOCAL_NIM_LLM_DEVICE_ID:
    env_overrides.append(f"LLM_DEVICE_ID={LOCAL_NIM_LLM_DEVICE_ID}")
if LOCAL_NIM_VLM_DEVICE_ID:
    env_overrides.append(f"VLM_DEVICE_ID={LOCAL_NIM_VLM_DEVICE_ID}")
if LOCAL_NIM_LLM_DEVICE_ID and LOCAL_NIM_VLM_DEVICE_ID:
    if LOCAL_NIM_LLM_DEVICE_ID == LOCAL_NIM_VLM_DEVICE_ID:
        env_overrides.extend(["LLM_MODE=local_shared", "VLM_MODE=local_shared"])
    else:
        env_overrides.extend(["LLM_MODE=local", "VLM_MODE=local"])

generate_args = {
    "profile": selected_profile,
    "env_overrides": env_overrides,
}
if selected_profile == "alerts":
    generate_args["alerts_mode"] = ALERTS_MODE
if NGC_CLI_API_KEY.strip():
    generate_args["ngc_cli_api_key"] = NGC_CLI_API_KEY
if NVIDIA_API_KEY.strip():
    generate_args["nvidia_api_key"] = NVIDIA_API_KEY

generate_result = require_success(
    tool_call(OrchestratorTool.DOCKER_GENERATE, mcp_url=MCP_URL, agent_dir=AGENT_DIR, arguments=generate_args),
    "docker_generate",
)
DOCKER_COMPOSE_ID = generate_result["docker_compose_id"]
print("DOCKER_COMPOSE_ID:", DOCKER_COMPOSE_ID)

### 7.4 Bring the generated deployment up

Run this after `DOCKER_COMPOSE_ID` has been created by the previous cell.

In [ ]:
if not globals().get("DOCKER_COMPOSE_ID"):
    raise RuntimeError("DOCKER_COMPOSE_ID is not set. Run the compose generation cell first.")

up_result = require_success(
    tool_call(
        OrchestratorTool.DOCKER_UP,
        mcp_url=MCP_URL,
        agent_dir=AGENT_DIR,
        arguments={"docker_compose_id": DOCKER_COMPOSE_ID},
    ),
    "docker_up",
)
DOCKER_UP_OPS_ID = up_result["docker_compose_ops_id"]
print("DOCKER_UP_OPS_ID:", DOCKER_UP_OPS_ID)


### 7.5 Check compose status

Run this cell to poll the compose operation until it finishes.

In [ ]:

if not globals().get("DOCKER_UP_OPS_ID"):
    raise RuntimeError("DOCKER_UP_OPS_ID is not set. Run the compose up cell first.")

UP_STATUS_RESULT = poll_compose_op(DOCKER_UP_OPS_ID, mcp_url=MCP_URL, agent_dir=AGENT_DIR)
print("Final deployment status:", UP_STATUS_RESULT["status"])
print("VSS UI:", build_vss_ui_url() or "unavailable")


## 8. Verify host reachability from inside the sandbox

<span style="color:red"><strong>Important:</strong> make sure VSS is already deployed on the host before running this step.</span>

Run the next cell after section 7 if you want to confirm the sandbox can reach host services through `host.openshell.internal`.

This checks the policy-allowed host ports and helps distinguish between:

- policy or DNS issues, and
- a host service that simply is not listening on the probed port.

In [ ]:
import shlex
import subprocess

HOST_PORTS = (7777, 3000, 8000, 9902, 30888, 5601, 6006, 9200, 8081, 31000)


def run(cmd, check=False):
    print("$", shlex.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True, check=check)
    if r.stdout:
        print(r.stdout)
    if r.stderr:
        print(r.stderr)
    return r


gateway_container = subprocess.run(
    ["docker", "ps", "--format", "{{.Names}}"], capture_output=True, text=True, check=True
).stdout.splitlines()
gateway_container = next((name for name in gateway_container if name.startswith("openshell-cluster-")), None)
print("Gateway container:", gateway_container)

if not gateway_container:
    raise RuntimeError("Could not determine the OpenShell gateway container for host reachability checks.")

host_probe_script = f"""
if command -v python3 >/dev/null 2>&1; then
  python3 - <<'PY'
import socket
host = {HOST_INTERNAL_ALIAS!r}
ports = {list(HOST_PORTS)}
for port in ports:
    s = socket.socket()
    s.settimeout(3)
    try:
        s.connect((host, port))
    except Exception as exc:
        print(f\"{{port}}: FAIL {{exc}}\")
    else:
        print(f\"{{port}}: OK\")
    finally:
        s.close()
PY
else
  echo \"python3 not available in sandbox for host probe\"
  exit 1
fi
""".strip()

print("Host service reachability from inside the sandbox:")
run(
    [
        "sudo",
        "docker",
        "exec",
        gateway_container,
        "kubectl",
        "exec",
        "-n",
        "openshell",
        NEMOCLAW_SANDBOX_NAME,
        "--",
        "sh",
        "-lc",
        host_probe_script,
    ]
)
print(
    f"Note: 'Connection refused' means the sandbox reached {HOST_INTERNAL_ALIAS}, "
    "but nothing is listening on that host/port combination."
)


## 9. Tear the deployment down (optional)

Use this when you want to stop and remove the generated deployment after the host reachability checks.

In [ ]:
if not globals().get("DOCKER_COMPOSE_ID"):
    raise RuntimeError("DOCKER_COMPOSE_ID is not set. Run the compose generation cell first.")

down_result = require_success(
    tool_call(
        OrchestratorTool.DOCKER_DOWN,
        mcp_url=MCP_URL,
        agent_dir=AGENT_DIR,
        arguments={"docker_compose_id": DOCKER_COMPOSE_ID},
    ),
    "docker_down",
)
DOCKER_DOWN_OPS_ID = down_result["docker_compose_ops_id"]
print("DOCKER_DOWN_OPS_ID:", DOCKER_DOWN_OPS_ID)

DOWN_STATUS_RESULT = poll_compose_op(DOCKER_DOWN_OPS_ID, mcp_url=MCP_URL, agent_dir=AGENT_DIR)
print("Final shutdown status:", DOWN_STATUS_RESULT["status"])


## 10. Uninstall NemoClaw (Optional)

In [ ]:
import subprocess

cmd = "curl -fsSL https://raw.githubusercontent.com/NVIDIA/NemoClaw/refs/heads/main/uninstall.sh | bash -s -- --yes"
print(f"$ {cmd}")
subprocess.run(["bash", "-lc", cmd], check=True)